In [ ]:
!pip -q install amplpy sympy numpy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 23.3 MB/s eta 0:00:00


In [ ]:
# 1. Instalación e inicialización de AMPL con tu licencia Community Edition
from amplpy import ampl_notebook

# Inicializa AMPL en Colab con tu UUID (Community Edition) Inicializar AMPL
ampl = ampl_notebook(
    modules=["highs", "cbc", "gurobi", "cplex"],  # solvers disponibles
    license_uuid="936b618d-a013-406f-9809-49679f557c26"
)

Licensed to AMPL Academic Community Edition License for <m.godoyseplveda@uandresbello.edu>.


In [ ]:
%%writefile lp_generic.mod
set ROWS;                      # restricciones
set COLS;                      # variables

param A {ROWS, COLS};          # matriz de coeficientes
param b {ROWS};                # RHS
param c {COLS};                # coeficientes del objetivo
param sense;                   #  +1 = Max ,  -1 = Mín

var x {COLS} >= 0;

maximize Z:  sense * sum {j in COLS} c[j] * x[j];

s.t. Con {i in ROWS}:
        sum {j in COLS} A[i,j] * x[j] <= b[i];      # todas “<=”

Overwriting lp_generic.mod


In [ ]:
%%writefile 6_3_1_primal.dat
set ROWS := R1 R2 ;
set COLS := x1 x2 ;

param sense := 1 ;              # MAX

param A :     x1  x2 :=
      R1      5   2
      R2      1   2 ;

param b := R1 20   R2 10 ;

param c := x1 6    x2 8 ;


Overwriting 6_3_1_primal.dat


In [ ]:
%%writefile 6_3_1_dual.dat
set ROWS := y1 y2 ;          # variables duales
set COLS := c1 c2 ;

param sense := -1 ;          # MIN (se implementa como Max de -Z)

# matriz A  (ya con signo menos para quedar en ≤)
param A :    c1   c2 :=
      y1    -5   -1
      y2    -2   -2 ;

# RHS tambien con signo menos
param b := y1 -6   y2 -8 ;

# coeficientes del objetivo (los RHS originales)
param c := c1 20   c2 10 ;



Overwriting 6_3_1_dual.dat


In [ ]:
ampl.reset()
ampl.read('lp_generic.mod')
ampl.readData('6_3_1_primal.dat')

ampl.option['solver'] = 'highs'
ampl.option['solver_msg'] = 0          # sin log extenso
ampl.solve()

print("⮞  Solución PRIMAL")
ampl.display('x','Z')

HiGHS 1.11.0: ⮞  Solución PRIMAL
x [*] :=
x1  2.5
x2  3.75
;

Z = 45



In [ ]:
ampl.reset()
ampl.read('lp_generic.mod')
ampl.readData('6_3_1_dual.dat')

ampl.option['solver'] = 'highs'
ampl.solve()

print("\n⮞  Solución DUAL")
ampl.display('x','Z')


HiGHS 1.11.0: 
⮞  Solución DUAL
x [*] :=
c1  0.5
c2  3.5
;

Z = -45



In [ ]:
print("\nTabla de soluciones básicas complementarias:")
primal = {'x1': 2.5, 'x2': 3.75}   # de la celda 10
dual   = {'c1': 0.5, 'c2': 3.5}    # de la celda 11

print("{:>6} {:>8} {:>8}".format("Var","Primal","Dual"))
for (pname,pval), (dname,dval) in zip(primal.items(), dual.items()):
    print("{:>6} {:>8} {:>8}".format(pname, pval, dval))



Tabla de soluciones básicas complementarias:
   Var   Primal     Dual
    x1      2.5      0.5
    x2     3.75      3.5


In [ ]:
ampl.reset()
ampl.read('lp_generic.mod')
ampl.readData('6_3_1_primal.dat')

ampl.option['solver'] = 'highs'
ampl.option['highs_options'] = 'log_dev_level=3'   # verbose iteraciones
ampl.solve()                                       # muestra pivotes
ampl.display('x')


HiGHS 1.11.0: x [*] :=
x1  0
x2  0
;

